In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到停止销售时间，所以这里需要PLM的生命周期全表

In [2]:
month_date = 202512
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2025-12-31')

productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}

### 读取物流数据，只保留3大渠道、并且是国内的数据

In [3]:
df = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾-包含停止发货.xlsx')
df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
print(len(df))
df.head()

498632


,商品编码,渠道,实际出库数量,发货月份,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态,物料号
0,1001001500116,零售,12,2025年10月,1001001500116,吸油烟机,3358,40296,Z8T,国内,停止销售,1001001500116
1,1009000600033,零售,1,2025年10月,1009000600033,蒸烤烹饪机,3450,3450,ZK50-02-F1,国内,量产,1009000600033
2,1009000500035,零售,3,2025年10月,1009000500035,灶蒸烤烹饪机,5280,15840,JZT-ZK46-X2,国内,量产,1009000500035
3,1001001500131,零售,6,2025年10月,1001001500131,吸油烟机,2988,17928,02-Z6TA,国内,停止销售,1001001500131
4,1002003700049,零售,5,2025年10月,1002003700049,灶具,2550,12750,H8B,国内,量产,1002003700049


In [4]:
# 长尾只看3大渠道。每个渠道的停止销售时间和既定时间的差距，所以只需要保留物料号、渠道、产品组、标准型号、国内/海外
df1 = df.copy()
df1 = df1[(df1['渠道'].isin(channel))&df1['国内/海外'].isin(['国内'])]
df1 = df1[['物料号','渠道','产品组','标准型号','国内/海外']].drop_duplicates().reset_index(drop=True)
df1

,物料号,渠道,产品组,标准型号,国内/海外
0,1001001500116,零售,吸油烟机,Z8T,国内
1,1009000600033,零售,蒸烤烹饪机,ZK50-02-F1,国内
2,1009000500035,零售,灶蒸烤烹饪机,JZT-ZK46-X2,国内
3,1001001500131,零售,吸油烟机,02-Z6TA,国内
4,1002003700049,零售,灶具,H8B,国内
...,...,...,...,...,...
1823,1013000100030,电商,家用净水机,YCZ-JT2500-01-H5S,国内
1824,1008000600008,电商,水槽洗碗机,JBSD2F-02-M5,国内
1825,1003000100117,电商,消毒柜,ZTD100S-02-X20.i,国内
1826,1003000900004,电商,消毒柜,ZTD115X-01-Y1.i,国内


### 读取PLM的相关数据

In [5]:
df_product_life = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx')
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).map(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life['渠道'] = df_product_life['下属渠道']
#####
df_product_life_map_df = df_product_life[['物料号','渠道','对应渠道状态','产品状态','产品型号','停止销售时间']]
df_product_life_map_df 

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,物料号,渠道,对应渠道状态,产品状态,产品型号,停止销售时间
4,1004000200090,NaN,NaN,停止发货,10T-JSG15-0606FR,NaN
5,1004000200079,NaN,NaN,停止发货,10T-JSG19-0607FR,NaN
6,1004000200084,NaN,NaN,停止发货,10T-JSG25-0608FR,NaN
7,1004000200035,NaN,NaN,停止发货,10T-JSQ16-0601,NaN
8,1004000200060,NaN,NaN,停止发货,10T-JSQ16-0601FR,NaN
...,...,...,...,...,...,...
9356,1001002200009,工程,在售,量产,iMES-45-C1,NaN
9357,1001002200005,工程,在售,量产,iMES-60-D1,NaN
9358,1001002200001,NaN,NaN,作废,iMES-C1,NaN
9359,1001002200007,工程,在售,量产,iMES-D1G-K1,NaN


### 保留各个渠道停止销售的产品

In [6]:
df_calu = pd.merge(df1,df_product_life_map_df,how='left',on=['物料号','渠道'])
df_calu['停止销售时间'] = pd.to_datetime(df_calu['停止销售时间'])
df_calu = df_calu[df_calu['渠道'].isin(channel)]
df_calu = df_calu[df_calu['对应渠道状态'].isin(['停止销售'])]
df_calu

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_25520\1975798613.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_calu['停止销售时间'] = pd.to_datetime(df_calu['停止销售时间'])


,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间
0,1001001500116,零售,吸油烟机,Z8T,国内,停止销售,停止销售,CXW-358-Z8T(不带罩),2025-11-24 16:51:46
3,1001001500131,零售,吸油烟机,02-Z6TA,国内,停止销售,停止销售,CXW-358-02-Z6TA(不带罩),2025-11-24 16:51:46
11,1002003700004,零售,灶具,02-HECB,国内,停止销售,停止销售,JZT-02-HE01CB-12T,2026-01-04 12:12:01
12,1001001500117,零售,吸油烟机,Z5TS,国内,停止销售,停止销售,CXW-358-Z5TS（不带罩）,2026-01-04 12:36:17
35,1004000500237,零售,热水器,JSQ25-H1303,国内,停止销售,量产,JSQ25-H1303-FR-12T,2025-11-14 15:02:45
...,...,...,...,...,...,...,...,...,...
1801,1009000500020,电商,灶蒸烤烹饪机,JZT-ZK42-02-X3A.i,国内,停止销售,停止销售,JZT-ZK42-02-X3A.i,2025-06-09 11:01:32
1807,1009000600001,电商,蒸烤烹饪机,ZK-T1.i,国内,停止销售,停止销售,ZK-T1.i,2025-05-27 11:13:44
1814,1002003400143,工程,灶具,TH7B,国内,停止销售,停止销售,JZT-TH7B(SH)-12T,2025-12-25 15:12:36
1819,1008000300023,电商,水槽洗碗机,JBSD2T-K3A,国内,停止销售,退市预警,JBSD2T-K3A,2025-07-23 15:02:58


In [7]:
from calendar import month
from pandas import DateOffset
#  只要有一个产品型号是长尾，那么这个标准型号就是长尾
for index,row in df_calu.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df_calu.loc[index,'是否长尾型号'] = '是'
df_calu['是否长尾型号'] = df_calu['是否长尾型号'].fillna('否')
df_calu

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间,是否长尾型号
0,1001001500116,零售,吸油烟机,Z8T,国内,停止销售,停止销售,CXW-358-Z8T(不带罩),2025-11-24 16:51:46,否
3,1001001500131,零售,吸油烟机,02-Z6TA,国内,停止销售,停止销售,CXW-358-02-Z6TA(不带罩),2025-11-24 16:51:46,否
11,1002003700004,零售,灶具,02-HECB,国内,停止销售,停止销售,JZT-02-HE01CB-12T,2026-01-04 12:12:01,否
12,1001001500117,零售,吸油烟机,Z5TS,国内,停止销售,停止销售,CXW-358-Z5TS（不带罩）,2026-01-04 12:36:17,否
35,1004000500237,零售,热水器,JSQ25-H1303,国内,停止销售,量产,JSQ25-H1303-FR-12T,2025-11-14 15:02:45,否
...,...,...,...,...,...,...,...,...,...,...
1801,1009000500020,电商,灶蒸烤烹饪机,JZT-ZK42-02-X3A.i,国内,停止销售,停止销售,JZT-ZK42-02-X3A.i,2025-06-09 11:01:32,否
1807,1009000600001,电商,蒸烤烹饪机,ZK-T1.i,国内,停止销售,停止销售,ZK-T1.i,2025-05-27 11:13:44,否
1814,1002003400143,工程,灶具,TH7B,国内,停止销售,停止销售,JZT-TH7B(SH)-12T,2025-12-25 15:12:36,否
1819,1008000300023,电商,水槽洗碗机,JBSD2T-K3A,国内,停止销售,退市预警,JBSD2T-K3A,2025-07-23 15:02:58,否


In [8]:
df_calu.to_excel(fr"C:\Users\zhangbon\Desktop\长尾明细.xlsx", index=False)


In [9]:
df_out = pd.DataFrame()
df_out['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'长尾标准型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'标准型号数量'] = vals
df_out['长尾标准型号占比'] = df_out['长尾标准型号数量']/df_out['标准型号数量']
df_out

,产品类别,长尾标准型号数量,标准型号数量,长尾标准型号占比
0,吸油烟机,24.0,75.0,0.320000
1,灶具,4.0,43.0,0.093023
2,蒸烤微合计,0.0,24.0,0.000000
3,灶集成,0.0,9.0,0.000000
4,消毒柜,1.0,9.0,0.111111
5,热水器,1.0,16.0,0.062500
6,净水机,4.0,8.0,0.500000
7,洗碗机,5.0,48.0,0.104167


In [10]:
df_out1 = pd.DataFrame()
df_out1['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'零售长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'工程长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商产品型号数量'] = vals2 
    df_out1.loc[df_out1['产品类别']==k,'电商长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    

    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'全渠道长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
df_out1

,产品类别,零售长尾产品型号数量,零售产品型号数量,零售长尾产品型号占比,工程长尾产品型号数量,工程产品型号数量,工程长尾产品型号占比,电商长尾产品型号数量,电商产品型号数量,电商长尾产品型号占比,全渠道长尾产品型号数量,全渠道产品型号数量,全渠道长尾产品型号占比
0,吸油烟机,6.0,36.0,0.166667,21.0,44.0,0.477273,0.0,33.0,0.0,27.0,92.0,0.293478
1,灶具,0.0,79.0,0.000000,4.0,42.0,0.095238,0.0,11.0,0.0,4.0,116.0,0.034483
2,蒸烤微合计,0.0,15.0,0.000000,0.0,12.0,0.000000,0.0,14.0,0.0,0.0,24.0,0.000000
3,灶集成,0.0,21.0,0.000000,0.0,1.0,0.000000,0.0,11.0,0.0,0.0,21.0,0.000000
4,消毒柜,1.0,8.0,0.125000,0.0,3.0,0.000000,0.0,6.0,0.0,1.0,11.0,0.090909
5,热水器,0.0,9.0,0.000000,1.0,12.0,0.083333,0.0,6.0,0.0,1.0,20.0,0.050000
6,净水机,4.0,10.0,0.400000,0.0,5.0,0.000000,0.0,2.0,0.0,4.0,10.0,0.400000
7,洗碗机,3.0,18.0,0.166667,2.0,23.0,0.086957,0.0,37.0,0.0,5.0,61.0,0.081967


In [11]:
with pd.ExcelWriter(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\长尾统计结果-保留了停止发货.xlsx') as writer:
    df_out.to_excel(writer,sheet_name='标准型号统计',index=False)
    df_out1.to_excel(writer,sheet_name='分渠道产品型号统计',index=False)
